# Analysis HIB Frequency Sweep

This notebook is the cleaned source notebook for the 7-series undervolting and impedance-oriented frequency sweep experiments.

## Purpose
- Sweep FPGA core voltage and clock frequency.
- Observe clock-health and register-state behavior during undervolting.
- Save the collected `points` list for later plotting in `Plot_results/`.


## How to run
1. Connect the CW-Lite and the FPGA target.
2. Update the bitstream path below.
3. Make sure the target exposes the same `target.REG_*` registers used in the helper functions.
4. Run the setup cells.
5. Run the sweep cell and save the resulting `points` list.

The saved sweep output can then be copied to `Plot_results/freq_volt3.txt` and reused for plotting.


In [ ]:
from pathlib import Path
import time

import chipwhisperer as cw
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

# Update this path for your local FPGA image.
BITSTREAM = Path("path/to/cw305_top.bit")

# Sweep configuration.
VOLTAGE_START = 1.00
VOLTAGE_STOP = 0.63
VOLTAGE_STEP = -0.01
FREQUENCIES_HZ = np.linspace(1e6, 150e6, 20)

# Save the final list to this file, then copy it to Plot_results/freq_volt3.txt.
OUTPUT_FILE = Path("freq_volt3.txt")


In [ ]:
# CW-Lite is used as the capture/control platform.
scope = cw.scope()
target = cw.target(scope, cw.targets.CW305, bsfile=str(BITSTREAM), force=False)

target.vccint_set(1.0)
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)

print("Target initialized. Update BITSTREAM if programming fails.")


In [ ]:
def debug_reg_reset():
    """Reset the experiment-side debug register and confirm it is readable."""
    target.fpga_write(target.REG_DEBUG_RST, [0x01])
    time.sleep(0.1)
    target.fpga_write(target.REG_DEBUG_RST, [0x00])
    debug_value = target.fpga_read(target.REG_DEBUG_VAL, 10)
    return debug_value


def configure_output_clock(freq_hz):
    """Route the requested test frequency to PLL output 1."""
    target.pll.pll_enable_set(True)
    target.pll.pll_outenable_set(False, 0)
    target.pll.pll_outenable_set(True, 1)
    target.pll.pll_outenable_set(False, 2)
    target.pll.pll_outfreq_set(freq_hz, 1)
    target.clkusbautooff = True
    target.clksleeptime = 1


def read_debug_snapshot():
    """Read the 10-byte debug payload and keep the raw fields for later plotting."""
    raw = target.fpga_read(target.REG_DEBUG_VAL, 10)
    return {
        "reg_assign": raw[0],
        "logic_assign": raw[1],
        "blocking_assign": raw[2],
        "clock_count": raw[3:10],
    }


In [ ]:
def run_frequency_voltage_sweep():
    """Sweep frequency and VCCINT, then record undervolt and recovery states."""
    points = []
    voltage_count = int(round((VOLTAGE_START - VOLTAGE_STOP) / abs(VOLTAGE_STEP))) + 1
    voltages = np.round(np.linspace(VOLTAGE_START, VOLTAGE_STOP, voltage_count), 2)

    for freq_hz in tqdm(FREQUENCIES_HZ, desc="Frequency sweep"):
        configure_output_clock(freq_hz)

        for volt in voltages:
            debug_reg_reset()

            target.vccint_set(float(volt))
            target.fpga_write(target.REG_SR_RST, [0x01])
            time.sleep(0.5)
            target.fpga_write(target.REG_SR_RST, [0x00])
            undervolt_snapshot = read_debug_snapshot()

            target.vccint_set(1.0)
            recovery_snapshot = read_debug_snapshot()

            points.append(
                {
                    "freq": float(freq_hz),
                    "volt": float(volt),
                    "clock_count": undervolt_snapshot["clock_count"],
                    "reg_assign": undervolt_snapshot["reg_assign"],
                    "logic_assign": undervolt_snapshot["logic_assign"],
                    "blocking_assign": undervolt_snapshot["blocking_assign"],
                    "clock_counter_rec": recovery_snapshot["clock_count"],
                    "reg_assign_rec": recovery_snapshot["reg_assign"],
                    "blocking_assign_rec": recovery_snapshot["blocking_assign"],
                }
            )

    return points


In [ ]:
# Uncomment the next line to run the full sweep.
# points = run_frequency_voltage_sweep()

# Save the generated list for plotting.
# OUTPUT_FILE.write_text(repr(points), encoding="utf-8")
# print(f"Saved {len(points)} sweep points to {OUTPUT_FILE.resolve()}")


In [ ]:
# Quick sanity plot after the sweep.
# Replace `points` with your loaded/exported list if you reopen the notebook later.
# clock_counts = [int.from_bytes(point["clock_count"], byteorder="little") for point in points]
# voltages = [point["volt"] for point in points]
# plt.figure(figsize=(8, 4))
# plt.plot(voltages[:len(clock_counts)], clock_counts, marker="o")
# plt.xlabel("Voltage (V)")
# plt.ylabel("Clock Count")
# plt.title("Quick Clock-Health Sanity Check")
# plt.grid(True, alpha=0.3)
# plt.show()
